In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("deliveries_cleaned.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "matchid", "inning", "over", "ball"]).reset_index(drop=True)

In [2]:
batting_match = df.groupby(
    ["matchid", "date", "inning", "batsman", "batting_team", "bowling_team"],
    as_index=False
).agg(
    runs_scored=("batsman_runs", "sum"),
    balls_faced=("iswide", lambda x: (x == 0).sum()),
    fours=("batsman_runs", lambda x: (x == 4).sum()),
    sixes=("batsman_runs", lambda x: (x == 6).sum()),
    dismissed=("is_wicket", "max")
)

batting_match = batting_match.sort_values(["batsman", "date", "matchid"]).reset_index(drop=True)

batting_match["strike_rate"] = np.where(
    batting_match["balls_faced"] > 0,
    batting_match["runs_scored"] / batting_match["balls_faced"] * 100,
    0
)

In [3]:
batting_match["career_matches"] = batting_match.groupby("batsman").cumcount()

batting_match["career_runs"] = (
    batting_match.groupby("batsman")["runs_scored"].cumsum().shift(1).fillna(0)
)
batting_match["career_balls"] = (
    batting_match.groupby("batsman")["balls_faced"].cumsum().shift(1).fillna(0)
)

# These were missing/truncated in the original
batting_match["career_avg_runs"] = np.where(
    batting_match["career_matches"] > 0,
    batting_match["career_runs"] / batting_match["career_matches"],
    0
)
batting_match["career_strike_rate"] = np.where(
    batting_match["career_balls"] > 0,
    batting_match["career_runs"] / batting_match["career_balls"] * 100,
    0
)

In [4]:
batting_match["form_runs_last_5"] = (
    batting_match.groupby("batsman")["runs_scored"]
    .shift(1).rolling(5, min_periods=1).mean()
)
batting_match["form_runs_last_10"] = (
    batting_match.groupby("batsman")["runs_scored"]
    .shift(1).rolling(10, min_periods=1).mean()
)
batting_match["form_sr_last_5"] = (
    batting_match.groupby("batsman")["strike_rate"]
    .shift(1).rolling(5, min_periods=1).mean()
)

In [5]:
# WRONG (original) — uses future data via transform("mean")
# batting_match["avg_runs_vs_opponent"] = batting_match.groupby(...)["runs_scored"].transform("mean")

# CORRECT — only uses past matches
batting_match["matches_vs_opponent"] = batting_match.groupby(["batsman", "bowling_team"]).cumcount()

batting_match["runs_vs_opponent_cumsum"] = (
    batting_match.groupby(["batsman", "bowling_team"])["runs_scored"]
    .cumsum().shift(1).fillna(0)
)

batting_match["avg_runs_vs_opponent"] = np.where(
    batting_match["matches_vs_opponent"] > 0,
    batting_match["runs_vs_opponent_cumsum"] / batting_match["matches_vs_opponent"],
    0
)

In [6]:
batting_match["is_first_innings"] = (batting_match["inning"] == 1).astype(int)
batting_match["is_rookie"] = (batting_match["career_matches"] < 5).astype(int)
batting_match["target_runs"] = batting_match["runs_scored"]

In [7]:
features_df = batting_match.drop(columns=[
    "runs_scored", "career_runs", "career_balls", "runs_vs_opponent_cumsum"
]).fillna(0)

features_df.to_csv("dataset_batting_features_v2.csv", index=False)
print("Done. Shape:", features_df.shape)

Done. Shape: (16439, 22)
